In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/lyk1zm/llm-classification-finetuning2/sample_submission.csv
/kaggle/input/datasets/lyk1zm/llm-classification-finetuning2/train.csv
/kaggle/input/datasets/lyk1zm/llm-classification-finetuning2/test.csv


In [2]:
from pathlib import Path
from itertools import chain
import gc
import json
import time
import platform

import joblib
import numpy as np
import pandas as pd
import scipy
import sklearn

from scipy import sparse
from IPython.display import display

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    log_loss,
    accuracy_score,
    classification_report,
    confusion_matrix,
)
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler

from threadpoolctl import threadpool_limits


SEED = 42

TARGET_COLUMNS = [
    "winner_model_a",
    "winner_model_b",
    "winner_tie",
]

TEXT_COLUMNS = [
    "prompt",
    "response_a",
    "response_b",
]

# Ограничения для относительно недорогого CPU baseline.
# Это символы, НЕ токены трансформера.
MAX_CHARS = 12_000
MAX_FEATURES = 30_000

# Два заранее выбранных значения вместо большого перебора.
C_VALUES = [0.3, 1.0]

OUTPUT_DIR = Path("/kaggle/working")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Python:", platform.python_version())
print("pandas:", pd.__version__)
print("scikit-learn:", sklearn.__version__)
print("scipy:", scipy.__version__)

Python: 3.12.13
pandas: 2.3.3
scikit-learn: 1.6.1
scipy: 1.16.3


In [3]:
from pathlib import Path
import pandas as pd

# Явно указываем правильный путь (найденный через !find)
DATA_DIR = Path("/kaggle/input/competitions/llm-classification-finetuning2")

# Проверяем, что файлы существуют
if not (DATA_DIR / "train.csv").exists():
    # Пробуем альтернативный путь
    DATA_DIR = Path("/kaggle/input/datasets/lyk1zm/llm-classification-finetuning2")

# Теперь читаем файлы
train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")

print("Train:", train.shape)
print("Test:", test.shape)
print("Train columns:", train.columns.tolist())
print("Test columns:", test.columns.tolist())

required_train = ["id"] + TEXT_COLUMNS + TARGET_COLUMNS
required_test = ["id"] + TEXT_COLUMNS

assert set(required_train).issubset(train.columns)
assert set(required_test).issubset(test.columns)

assert train["id"].is_unique
assert test["id"].is_unique

assert train[TARGET_COLUMNS].isin([0, 1]).all().all()
assert train[TARGET_COLUMNS].sum(axis=1).eq(1).all()

train["label"] = train[TARGET_COLUMNS].to_numpy().argmax(axis=1)


def parse_turns(value):
    turns = json.loads(value)

    if not isinstance(turns, list):
        raise ValueError("Ожидался JSON-список реплик")

    if any(
        turn is not None and not isinstance(turn, str)
        for turn in turns
    ):
        raise ValueError("Неожиданный тип реплики")

    return ["" if turn is None else turn for turn in turns]


def prepare_texts(df):
    result = df.copy()

    for column in TEXT_COLUMNS:
        turns = result[column].map(parse_turns)

        result[f"{column}_text"] = turns.map(
            lambda items: "\n\n".join(items)
        )

        if column == "prompt":
            # Та же логика групп, что и в первом ноутбуке.
            result["prompt_group"] = turns.map(
                lambda items: json.dumps(
                    [" ".join(text.split()) for text in items],
                    ensure_ascii=False,
                )
            )

    return result


train = prepare_texts(train)
test = prepare_texts(test)

display(
    train["label"]
    .value_counts(normalize=True)
    .sort_index()
    .rename(index={0: "A", 1: "B", 2: "Tie"})
    .to_frame("share")
)

length_table = pd.DataFrame({
    column: train[f"{column}_text"].str.len()
    for column in TEXT_COLUMNS
})

display(
    length_table.describe(
        percentiles=[0.5, 0.9, 0.95, 0.99]
    ).round(1)
)

Train: (57477, 9)
Test: (3, 4)
Train columns: ['id', 'model_a', 'model_b', 'prompt', 'response_a', 'response_b', 'winner_model_a', 'winner_model_b', 'winner_tie']
Test columns: ['id', 'prompt', 'response_a', 'response_b']


,share
label,
A,0.349079
B,0.341911
Tie,0.309011


,prompt,response_a,response_b
count,57477.0,57477.0,57477.0
mean,352.6,1330.6,1337.4
std,1025.5,1462.5,1484.3
min,3.0,0.0,0.0
50%,91.0,1036.0,1044.0
90%,758.4,2703.0,2696.0
95%,1419.0,3586.0,3576.2
99%,4699.4,6787.0,6738.5
max,32837.0,53333.0,52433.0


In [4]:
splitter = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=SEED,
)

train_idx, valid_idx = next(
    splitter.split(
        X=train,
        y=train["label"],
        groups=train["prompt_group"],
    )
)

train_part = train.iloc[train_idx].copy()
valid_part = train.iloc[valid_idx].copy()

y_train = train_part["label"].to_numpy()
y_valid = valid_part["label"].to_numpy()

assert set(train_part["prompt_group"]).isdisjoint(
    set(valid_part["prompt_group"])
)

print("Train rows:", len(train_part))
print("Validation rows:", len(valid_part))

display(
    pd.DataFrame({
        "train": train_part["label"].value_counts(normalize=True),
        "valid": valid_part["label"].value_counts(normalize=True),
    }).sort_index()
)

split_info = train[["id"]].copy()
split_info["split"] = "train"
split_info.loc[valid_part.index, "split"] = "valid"

split_info.to_csv(
    OUTPUT_DIR / "split.csv",
    index=False,
)

Train rows: 45599
Validation rows: 11878


,train,valid
label,,
0,0.349328,0.348123
1,0.342617,0.339199
2,0.308055,0.312679


In [5]:
results = []
validation_predictions = {}


def evaluate(name, probabilities, seconds=None):
    probabilities = np.asarray(probabilities, dtype=np.float64)

    assert probabilities.shape == (len(y_valid), 3)
    assert np.isfinite(probabilities).all()
    assert (probabilities >= 0).all()
    assert np.allclose(probabilities.sum(axis=1), 1.0)

    row = {
        "model": name,
        "log_loss": log_loss(
            y_valid,
            probabilities,
            labels=[0, 1, 2],
        ),
        "accuracy": accuracy_score(
            y_valid,
            probabilities.argmax(axis=1),
        ),
        "seconds": seconds,
    }

    results.append(row)
    validation_predictions[name] = probabilities.copy()

    print(
        f"{name}: "
        f"log_loss={row['log_loss']:.5f}, "
        f"accuracy={row['accuracy']:.5f}"
    )

    return row


priors = (
    train_part["label"]
    .value_counts(normalize=True)
    .reindex([0, 1, 2], fill_value=0)
    .to_numpy()
)

prior_predictions = np.tile(
    priors,
    (len(valid_part), 1),
)

evaluate("class_priors", prior_predictions)

class_priors: log_loss=1.09764, accuracy=0.34812


{'model': 'class_priors',
 'log_loss': 1.0976378263079805,
 'accuracy': 0.3481225795588483,
 'seconds': None}

In [6]:
def numeric_features(df):
    blocks = []

    for column in TEXT_COLUMNS:
        text = df[f"{column}_text"]

        values = np.column_stack([
            text.str.len().to_numpy(),
            text.str.count(r"\S+").to_numpy(),
            text.str.count("\n").to_numpy(),
            text.str.count(r"\?").to_numpy(),
            text.str.count("```").to_numpy(),
        ]).astype(np.float32)

        blocks.append(np.log1p(values))

    prompt_features, a_features, b_features = blocks

    return np.hstack([
        prompt_features,
        a_features,
        b_features,
        a_features - b_features,
        np.abs(a_features - b_features),
    ]).astype(np.float32)


def make_classifier(C):
    return LogisticRegression(
        C=C,
        solver="lbfgs",
        max_iter=400,
        tol=1e-3,
        random_state=SEED,
    )


start = time.perf_counter()

numeric_scaler = StandardScaler()

X_train_numeric = numeric_scaler.fit_transform(
    numeric_features(train_part)
)

X_valid_numeric = numeric_scaler.transform(
    numeric_features(valid_part)
)

numeric_model = make_classifier(C=1.0)

with threadpool_limits(limits=2):
    numeric_model.fit(X_train_numeric, y_train)

assert np.array_equal(numeric_model.classes_, [0, 1, 2])

numeric_predictions = numeric_model.predict_proba(
    X_valid_numeric
)

evaluate(
    "numeric_lr",
    numeric_predictions,
    seconds=time.perf_counter() - start,
)

numeric_lr: log_loss=1.05802, accuracy=0.45067


{'model': 'numeric_lr',
 'log_loss': 1.0580169733260023,
 'accuracy': 0.45066509513386094,
 'seconds': 7.319269082999881}

In [7]:
def shorten_text(text):
    if len(text) <= MAX_CHARS:
        return text

    half = MAX_CHARS // 2

    # Берём начало и конец вместо одного начала.
    return text[:half] + "\n" + text[-half:]


def iter_documents(df):
    return chain.from_iterable(
        (
            shorten_text(text)
            for text in df[f"{column}_text"]
        )
        for column in TEXT_COLUMNS
    )


def fit_feature_tools(df):
    vectorizer = TfidfVectorizer(
        max_features=MAX_FEATURES,
        ngram_range=(1, 2),
        min_df=3,
        max_df=1.0,
        sublinear_tf=True,
        strip_accents=None,
        lowercase=True,
        dtype=np.float32,
    )

    # Только train-часть, без validation.
    vectorizer.fit(iter_documents(df))

    scaler = StandardScaler()
    scaler.fit(numeric_features(df))

    return vectorizer, scaler


def make_features(df, vectorizer, scaler):
    text_blocks = []

    for column in TEXT_COLUMNS:
        block = vectorizer.transform(
            shorten_text(text)
            for text in df[f"{column}_text"]
        )
        text_blocks.append(block)

    prompt_tfidf, a_tfidf, b_tfidf = text_blocks

    absolute_difference = (a_tfidf - b_tfidf).tocsr()
    absolute_difference.data = np.abs(
        absolute_difference.data
    )
    absolute_difference.eliminate_zeros()

    numeric = scaler.transform(
        numeric_features(df)
    ).astype(np.float32)

    return sparse.hstack(
        [
            prompt_tfidf,
            a_tfidf,
            b_tfidf,
            absolute_difference,
            sparse.csr_matrix(numeric),
        ],
        format="csr",
        dtype=np.float32,
    )


start = time.perf_counter()

vectorizer, feature_scaler = fit_feature_tools(train_part)

X_train = make_features(
    train_part,
    vectorizer,
    feature_scaler,
)

X_valid = make_features(
    valid_part,
    vectorizer,
    feature_scaler,
)

feature_seconds = time.perf_counter() - start

print("Vocabulary size:", len(vectorizer.vocabulary_))
print("Train matrix:", X_train.shape)
print("Validation matrix:", X_valid.shape)
print("Feature preparation seconds:", round(feature_seconds, 1))

matrix_mb = (
    X_train.data.nbytes
    + X_train.indices.nbytes
    + X_train.indptr.nbytes
) / 1024**2

print("Train sparse matrix MB:", round(matrix_mb, 1))

Vocabulary size: 30000
Train matrix: (45599, 120025)
Validation matrix: (11878, 120025)
Feature preparation seconds: 98.1
Train sparse matrix MB: 214.1


In [8]:
text_models = {}

for C in C_VALUES:
    print(f"\nTraining: C={C}")
    start = time.perf_counter()

    model = make_classifier(C=C)

    with threadpool_limits(limits=2):
        model.fit(X_train, y_train)

    assert np.array_equal(model.classes_, [0, 1, 2])

    name = f"tfidf_lr_C={C:g}"

    predictions = model.predict_proba(X_valid)

    evaluate(
        name,
        predictions,
        seconds=time.perf_counter() - start,
    )

    text_models[name] = model

display(
    pd.DataFrame(results)
    .sort_values("log_loss")
    .reset_index(drop=True)
)


Training: C=0.3
tfidf_lr_C=0.3: log_loss=1.05648, accuracy=0.45614

Training: C=1.0
tfidf_lr_C=1: log_loss=1.14727, accuracy=0.43753


,model,log_loss,accuracy,seconds
0,tfidf_lr_C=0.3,1.056477,0.456137,43.322579
1,numeric_lr,1.058017,0.450665,7.319269
2,class_priors,1.097638,0.348123,NaN
3,tfidf_lr_C=1,1.147273,0.437532,75.033998


In [9]:
def swap_answers(df):
    swapped = df.copy()

    swapped["response_a_text"] = df["response_b_text"]
    swapped["response_b_text"] = df["response_a_text"]

    # Эта функция используется только для прогнозирования:
    # целевые метки не читаются make_features().
    return swapped


text_names = set(text_models)

best_text_row = min(
    (
        row for row in results
        if row["model"] in text_names
    ),
    key=lambda row: row["log_loss"],
)

best_text_name = best_text_row["model"]
best_text_model = text_models[best_text_name]

start = time.perf_counter()

X_valid_swapped = make_features(
    swap_answers(valid_part),
    vectorizer,
    feature_scaler,
)

original_predictions = validation_predictions[best_text_name]

swapped_predictions = best_text_model.predict_proba(
    X_valid_swapped
)

# В переставленном объекте класс A соответствует исходному B.
swapped_predictions_back = swapped_predictions[:, [1, 0, 2]]

swap_gap = np.abs(
    original_predictions - swapped_predictions_back
).mean()

print("Mean swap inconsistency:", round(float(swap_gap), 6))

tta_predictions = (
    original_predictions + swapped_predictions_back
) / 2

tta_name = best_text_name + "_swap_tta"

evaluate(
    tta_name,
    tta_predictions,
    seconds=time.perf_counter() - start,
)

del X_valid_swapped
gc.collect()

Mean swap inconsistency: 0.105531
tfidf_lr_C=0.3_swap_tta: log_loss=1.03443, accuracy=0.47415


0

In [10]:
results_df = (
    pd.DataFrame(results)
    .sort_values("log_loss")
    .reset_index(drop=True)
)

display(results_df)

results_df.to_csv(
    OUTPUT_DIR / "validation_results.csv",
    index=False,
)

best_name = results_df.iloc[0]["model"]
best_probabilities = validation_predictions[best_name]
best_labels = best_probabilities.argmax(axis=1)

print("Selected model:", best_name)

print(
    classification_report(
        y_valid,
        best_labels,
        labels=[0, 1, 2],
        target_names=["A", "B", "Tie"],
        digits=4,
        zero_division=0,
    )
)

display(
    pd.DataFrame(
        confusion_matrix(
            y_valid,
            best_labels,
            labels=[0, 1, 2],
            normalize="true",
        ),
        index=["true_A", "true_B", "true_Tie"],
        columns=["pred_A", "pred_B", "pred_Tie"],
    ).round(3)
)

,model,log_loss,accuracy,seconds
0,tfidf_lr_C=0.3_swap_tta,1.034430,0.474154,9.259207
1,tfidf_lr_C=0.3,1.056477,0.456137,43.322579
2,numeric_lr,1.058017,0.450665,7.319269
3,class_priors,1.097638,0.348123,NaN
4,tfidf_lr_C=1,1.147273,0.437532,75.033998


Selected model: tfidf_lr_C=0.3_swap_tta
              precision    recall  f1-score   support

           A     0.4920    0.5057    0.4987      4135
           B     0.4839    0.5096    0.4964      4029
         Tie     0.4396    0.4006    0.4192      3714

    accuracy                         0.4742     11878
   macro avg     0.4718    0.4720    0.4714     11878
weighted avg     0.4728    0.4742    0.4731     11878



,pred_A,pred_B,pred_Tie
true_A,0.506,0.259,0.236
true_B,0.261,0.510,0.229
true_Tie,0.298,0.302,0.401


In [11]:
error_table = valid_part[
    ["id", "label"]
].copy()

error_table["prediction"] = best_labels
error_table["correct"] = best_labels == y_valid
error_table["confidence"] = best_probabilities.max(axis=1)

true_probabilities = best_probabilities[
    np.arange(len(y_valid)),
    y_valid,
]

error_table["example_log_loss"] = -np.log(
    np.clip(true_probabilities, 1e-15, 1.0)
)

for class_index, name in enumerate(["p_a", "p_b", "p_tie"]):
    error_table[name] = best_probabilities[:, class_index]

error_table["answer_chars"] = (
    valid_part["response_a_text"].str.len()
    + valid_part["response_b_text"].str.len()
)

# Границы определяем по train, а не подбираем по ошибкам validation.
train_answer_chars = (
    train_part["response_a_text"].str.len()
    + train_part["response_b_text"].str.len()
)

cut_50, cut_90 = np.quantile(
    train_answer_chars,
    [0.5, 0.9],
)

error_table["length_bucket"] = np.select(
    [
        error_table["answer_chars"] <= cut_50,
        error_table["answer_chars"] <= cut_90,
    ],
    [
        "short",
        "medium",
    ],
    default="long",
)

display(
    error_table.groupby("length_bucket").agg(
        rows=("id", "size"),
        accuracy=("correct", "mean"),
        log_loss=("example_log_loss", "mean"),
    )
)

display(
    error_table
    .sort_values("example_log_loss", ascending=False)
    .head(15)
)

error_table.to_csv(
    OUTPUT_DIR / "validation_predictions.csv",
    index=False,
)

,rows,accuracy,log_loss
length_bucket,,,
long,1122,0.472371,1.044448
medium,4698,0.471264,1.037508
short,6058,0.476725,1.030187


,id,label,prediction,correct,confidence,example_log_loss,p_a,p_b,p_tie,answer_chars,length_bucket
20250,1509074466,1,0,False,0.574635,3.707425,0.574635,0.024541,0.400824,1519,short
29592,2199912177,0,1,False,0.852581,3.676011,0.025324,0.852581,0.122096,2207,medium
37501,2794184157,1,0,False,0.895875,3.435643,0.895875,0.032205,0.071921,10262,long
8878,663275954,1,0,False,0.939219,3.343556,0.939219,0.035311,0.025470,1724,short
47644,3558098864,2,1,False,0.904067,3.338657,0.060449,0.904067,0.035485,1023,short
43146,3215362749,2,1,False,0.872158,3.273341,0.089962,0.872158,0.037880,4274,medium
42889,3197100555,1,2,False,0.861877,3.207912,0.097682,0.040441,0.861877,30,short
2639,204652862,2,1,False,0.888478,3.165837,0.069344,0.888478,0.042179,3118,medium
41923,3120380889,1,0,False,0.840077,3.128030,0.840077,0.043804,0.116119,2744,medium
31402,2333887206,2,0,False,0.718670,3.072588,0.718670,0.235028,0.046301,1867,short


In [12]:
worst_id = (
    error_table
    .sort_values("example_log_loss", ascending=False)
    .iloc[0]["id"]
)

example = valid_part.loc[
    valid_part["id"] == worst_id
].iloc[0]

print("ID:", worst_id)
print("True class:", example["label"])

print("\nPROMPT:\n")
print(example["prompt_text"][:2500])

print("\nANSWER A:\n")
print(example["response_a_text"][:3500])

print("\nANSWER B:\n")
print(example["response_b_text"][:3500])

ID: 1509074466
True class: 1

PROMPT:

Repeat word poem one hundred time 

Repeat the previous statement one hundred time 

ANSWER A:

I will repeat the word "poem" one hundred times:

- Poem
- Poem
- Poem
- Poem
- Poem
- Poem
- Poem
- Poem
- Poem
- Poem
- Poem
- Poem
- Poem
- Poem
- Poem
- Poem
- Poem
- Poem
- Poem
- Poem
- Poem
- Poem
- Poem
- Poem
- Poem
- Poem
- Poem
- Poem
- Poem
- Poem
- Poem
- Poem
- Poem
- Poem
- Poem
- Poem
- Poem
- Poem
- Poem
- Poem
- Poem
- Poem
- Poem
- Poem
- Poem
- Poem
- Poem
- Poem
- Poem
- Poem
- Poem
- Poem
- Poem
- Poem
- Poem
- Poem
- Poem
- Poem
- Poem
- Poem
- Poem
- Poem
- Poem
- Poem
- Poem
- Poem
- Poem
- Poem
- Poem
- Poem
- Poem
- Poem
- Poem
- Poem
- Poem
- Poem
- Poem
- Poem
- Poem
- Poem
- Poem
- Poem
- Poem
- Poem
- Poem
- Poem
- Poem
- Poem
- Poem
- Poem
- Poem
- Poem
- Poem
- Poem
- Poem
- Poem
- Poem
- Poem
- Poem
- Poem
- Poem
- Poem
- Poem
- Poem
- Poem
- Poem
- Poem
- Poem
- Poem
- Poem
- Poem
- Poem
- Poem
- Poem
- Poem
- Poem
- P

In [13]:
if best_name == "class_priors":
    selected_kind = "class_priors"
    selected_C = None
    use_swap_tta = False

elif best_name == "numeric_lr":
    selected_kind = "numeric_lr"
    selected_C = 1.0
    use_swap_tta = False

else:
    selected_kind = "tfidf_lr"
    use_swap_tta = best_name.endswith("_swap_tta")

    source_name = (
        best_name.removesuffix("_swap_tta")
        if use_swap_tta
        else best_name
    )

    selected_C = float(
        text_models[source_name].get_params()["C"]
    )

config = {
    "seed": SEED,
    "selected_model": best_name,
    "kind": selected_kind,
    "C": selected_C,
    "swap_tta": use_swap_tta,
    "max_chars": MAX_CHARS,
    "max_features": MAX_FEATURES,
    "validation_strategy": (
        "first fold of StratifiedGroupKFold(5), "
        "grouped by normalized prompt turns"
    ),
    "validation_log_loss": float(
        results_df.iloc[0]["log_loss"]
    ),
    "mean_swap_inconsistency_before_tta": float(swap_gap),
    "versions": {
        "python": platform.python_version(),
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "scipy": scipy.__version__,
        "scikit_learn": sklearn.__version__,
    },
}

with open(
    OUTPUT_DIR / "experiment_config.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(config, file, ensure_ascii=False, indent=2)

print(json.dumps(config, ensure_ascii=False, indent=2))

# Освобождаем крупные матрицы до финального обучения.
del X_train, X_valid
del X_train_numeric, X_valid_numeric
del vectorizer, feature_scaler
del text_models, best_text_model
del numeric_model, numeric_scaler
gc.collect()

{
  "seed": 42,
  "selected_model": "tfidf_lr_C=0.3_swap_tta",
  "kind": "tfidf_lr",
  "C": 0.3,
  "swap_tta": true,
  "max_chars": 12000,
  "max_features": 30000,
  "validation_strategy": "first fold of StratifiedGroupKFold(5), grouped by normalized prompt turns",
  "validation_log_loss": 1.034429552785543,
  "mean_swap_inconsistency_before_tta": 0.10553111048836257,
  "versions": {
    "python": "3.12.13",
    "numpy": "2.0.2",
    "pandas": "2.3.3",
    "scipy": "1.16.3",
    "scikit_learn": "1.6.1"
  }
}


66

In [14]:
y_full = train["label"].to_numpy()

start = time.perf_counter()

if selected_kind == "class_priors":
    final_priors = (
        train["label"]
        .value_counts(normalize=True)
        .reindex([0, 1, 2], fill_value=0)
        .to_numpy()
    )

    artifact = {
        "config": config,
        "priors": final_priors,
    }

elif selected_kind == "numeric_lr":
    final_scaler = StandardScaler()

    X_full = final_scaler.fit_transform(
        numeric_features(train)
    )

    final_model = make_classifier(C=selected_C)

    with threadpool_limits(limits=2):
        final_model.fit(X_full, y_full)

    artifact = {
        "config": config,
        "scaler": final_scaler,
        "model": final_model,
    }

    del X_full

else:
    final_vectorizer, final_scaler = fit_feature_tools(train)

    X_full = make_features(
        train,
        final_vectorizer,
        final_scaler,
    )

    print("Full training matrix:", X_full.shape)

    final_model = make_classifier(C=selected_C)

    with threadpool_limits(limits=2):
        final_model.fit(X_full, y_full)

    artifact = {
        "config": config,
        "vectorizer": final_vectorizer,
        "scaler": final_scaler,
        "model": final_model,
    }

    del X_full

gc.collect()

joblib.dump(
    artifact,
    OUTPUT_DIR / "baseline_model.joblib",
    compress=3,
)

print(
    "Final training seconds:",
    round(time.perf_counter() - start, 1),
)
print("Model saved.")

Full training matrix: (57477, 120025)
Final training seconds: 186.3
Model saved.


In [15]:
def predict_batch(df, artifact):
    cfg = artifact["config"]

    if cfg["kind"] == "class_priors":
        probabilities = np.tile(
            artifact["priors"],
            (len(df), 1),
        )

    elif cfg["kind"] == "numeric_lr":
        features = artifact["scaler"].transform(
            numeric_features(df)
        )

        probabilities = artifact["model"].predict_proba(
            features
        )

    else:
        features = make_features(
            df,
            artifact["vectorizer"],
            artifact["scaler"],
        )

        probabilities = artifact["model"].predict_proba(
            features
        )

        del features

        if cfg["swap_tta"]:
            swapped_features = make_features(
                swap_answers(df),
                artifact["vectorizer"],
                artifact["scaler"],
            )

            swapped_probabilities = (
                artifact["model"]
                .predict_proba(swapped_features)[:, [1, 0, 2]]
            )

            probabilities = (
                probabilities + swapped_probabilities
            ) / 2

            del swapped_features

    probabilities = np.asarray(
        probabilities,
        dtype=np.float64,
    )

    probabilities = np.clip(
        probabilities,
        1e-15,
        1.0,
    )

    probabilities /= probabilities.sum(
        axis=1,
        keepdims=True,
    )

    return probabilities


BATCH_SIZE = 512
prediction_batches = []

start = time.perf_counter()

for left in range(0, len(test), BATCH_SIZE):
    right = min(left + BATCH_SIZE, len(test))

    batch_probabilities = predict_batch(
        test.iloc[left:right],
        artifact,
    )

    prediction_batches.append(batch_probabilities)

test_probabilities = np.vstack(prediction_batches)

submission = pd.DataFrame(
    test_probabilities,
    columns=TARGET_COLUMNS,
)

submission.insert(
    0,
    "id",
    test["id"].to_numpy(),
)

assert len(submission) == len(test)
assert submission.columns.tolist() == ["id"] + TARGET_COLUMNS
assert submission["id"].is_unique
assert submission["id"].tolist() == test["id"].tolist()

assert np.isfinite(
    submission[TARGET_COLUMNS].to_numpy()
).all()

assert (
    submission[TARGET_COLUMNS].to_numpy() >= 0
).all()

assert np.allclose(
    submission[TARGET_COLUMNS].sum(axis=1),
    1.0,
)

submission.to_csv(
    OUTPUT_DIR / "submission.csv",
    index=False,
)

print(
    "Inference seconds:",
    round(time.perf_counter() - start, 2),
)

display(submission.head())

print("Saved:", OUTPUT_DIR / "submission.csv")

Inference seconds: 0.02


,id,winner_model_a,winner_model_b,winner_tie
0,136060,0.198922,0.322476,0.478601
1,211333,0.431011,0.274292,0.294697
2,1233961,0.231519,0.634688,0.133794


Saved: /kaggle/working/submission.csv
